<a href="https://colab.research.google.com/github/oktaforai-okta/ProGearSalesAI/blob/main/notebooks/progear-inventory-authorization-story.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One AI agent, different permissions
## A beginner-friendly ProGear inventory authorization story

This notebook explains why the **same ProGear Sales Agent** may allow one employee to update inventory while preventing another employee from doing so.

### What you will learn

- Why the user and the AI agent need separate identities
- Why reading inventory and changing inventory are different permissions
- How an Okta authorization policy makes the final allow-or-deny decision
- Why a denial means the security control is working, not that the AI agent is broken
- How to explain the outcome without exposing technical implementation details


## 1. Start with the business story

ProGear employees use one conversational assistant: the **ProGear Sales Agent**. A person might ask it to check stock, prepare a quote, or update an inventory count.

The employee does not need to know which business service stores inventory. They only need to know whether their requested action is allowed.

For this scenario:

| Employee | Job responsibility | Inventory access |
|---|---|---|
| **Sarah Sales** | Helps customers and checks product availability | Can **read** inventory, but cannot **increase** it |
| **Mike Manager** | Manages warehouse stock | Can **read** and **increase** inventory |

Both people use the same AI agent. The difference is the employee's identity, group membership, and requested action.


## 2. One-agent mental model

Think of the ProGear Sales Agent as a helpful employee standing at a secured service desk.

1. The employee signs in.
2. The employee asks the agent to perform an action.
3. The agent identifies itself and says which employee it is acting for.
4. Okta checks whether that employee is allowed to perform that action.
5. The business system receives a narrowly scoped token only when the policy allows it.

The agent does not decide that Sarah is less trusted than Mike. Okta applies ProGear's centrally managed policy consistently.


## 3. Read and write are separate permissions

A request to **check inventory** needs `inventory:read`. A request to **increase inventory** needs `inventory:write`.

This separation follows the principle of least privilege: give a person only the access required for their job. Sarah can answer a customer asking whether an item is available, but she cannot change warehouse records. Mike can do both because warehouse updates are part of his role.

| User and request | Permission needed | Policy result | User experience |
|---|---|---|---|
| Sarah checks stock | `inventory:read` | Allowed | Inventory can be shown |
| Sarah increases stock | `inventory:write` | Denied | Clear guidance to contact her manager |
| Mike checks stock | `inventory:read` | Allowed | Inventory can be shown |
| Mike increases stock | `inventory:write` | Allowed | Update may proceed through any additional safeguards |


## 4. Try the policy safely

The next cell is a **local simulation**. It does not connect to Okta, does not use a password or token, and does not change inventory.

If you are in Colab, choose an employee and action from the dropdowns, then select the play button on the left side of the cell.


In [ ]:
# @title Choose an employee and inventory action
employee = "Sarah Sales" # @param ["Sarah Sales", "Mike Manager"]
action = "Increase inventory" # @param ["Check inventory", "Increase inventory"]

APPROVED_DENIAL = (
    "I can’t increase inventory with your current permissions. "
    "Please contact your manager for assistance or approval."
)

POLICY = {
    "Sarah Sales": {"inventory:read"},
    "Mike Manager": {"inventory:read", "inventory:write"},
}
ACTION_TO_PERMISSION = {
    "Check inventory": "inventory:read",
    "Increase inventory": "inventory:write",
}

required_permission = ACTION_TO_PERMISSION[action]
allowed = required_permission in POLICY[employee]

print(f"Employee: {employee}")
print(f"Requested action: {action}")
print(f"Permission required: {required_permission}")
print(f"Decision: {'ALLOWED' if allowed else 'DENIED'}")
print()
if allowed:
    print("The ProGear Sales Agent may continue with this request.")
else:
    print(APPROVED_DENIAL)


Employee: Sarah Sales
Requested action: Increase inventory
Permission required: inventory:write
Decision: DENIED

I can’t increase inventory with your current permissions. Please contact your manager for assistance or approval.


### Expected result for Sarah's write request

```text
Employee: Sarah Sales
Requested action: Increase inventory
Permission required: inventory:write
Decision: DENIED

I can’t increase inventory with your current permissions. Please contact your manager for assistance or approval.
```

Notice what the message does **not** say. It does not mention an internal service, a routing component, or another agent. Those details are useful to administrators, but they do not help the employee decide what to do next.


## 5. What happens behind the scenes

Here is the authorization sequence in plain English:

```text
Sarah or Mike
     │ signs in and asks for an inventory action
     ▼
ProGear Sales Agent
     │ asks Okta to act for the signed-in employee
     ▼
Okta binds the employee and agent identities
     │ evaluates the requested permission
     ▼
Inventory authorization policy
     ├─ allowed → issue a short-lived, narrowly scoped token
     └─ denied  → do not issue the token; return helpful guidance
```

The important security property is that the business service is called only after Okta has issued the permission needed for that specific action.


## 6. ID-JAG, explained without the jargon

ID-JAG stands for **Identity Assertion JWT Authorization Grant**. You do not need to memorize that name.

For this story, ID-JAG is a short-lived, signed proof that says:

> This approved ProGear Sales Agent is acting on behalf of this signed-in employee.

Okta then uses that proof when it evaluates the Inventory policy. The final access token contains only the permission approved for the request. If `inventory:write` is not permitted, Okta does not issue a write token.


## 7. Optional: inspect an illustrative token

This example is fabricated for learning. It is not a live token and cannot access anything. Run the cell only if you want to see how the important facts can be represented.


In [ ]:
import json

illustrative_claims = {
    "subject": "signed-in employee",
    "actor": "ProGear Sales Agent",
    "audience": "ProGear Inventory API",
    "permissions": ["inventory:read"],
    "expires_in": "a short time",
}

print(json.dumps(illustrative_claims, indent=2))
print("\nThis is explanatory data, not a real credential.")


## 8. Optional: check that the public demo service is available

This read-only check calls the public health endpoint. It does not sign in, reveal data, or change inventory. If your environment blocks outbound internet access, skip this section.


In [ ]:
import json
from urllib.error import URLError
from urllib.request import urlopen

health_url = "https://progearsalesai-p2wm.onrender.com/health"

try:
    with urlopen(health_url, timeout=30) as response:
        health = json.load(response)
    print("Demo service status:", health.get("status", "unknown"))
    print("This was a read-only availability check.")
except URLError as error:
    print("The optional health check could not reach the demo service.")
    print("You can continue reading the notebook; the authorization lesson is self-contained.")


## 9. Try the live user experience

Use the [ProGear Sales Agent demo](https://progear-sales-aiagent.vercel.app) only if your administrator has given you a demo account.

1. Sign in as the sales user supplied by your administrator.
2. Ask: **How many units are in inventory?**
3. Observe that the read request can succeed.
4. Ask: **Increase inventory by 10 units.**
5. Observe the clear permission message.
6. Sign out completely.
7. Sign in as the warehouse manager supplied by your administrator.
8. Repeat the questions and observe the different authorization outcome.

> **Safety note:** Do not paste passwords, session cookies, private keys, or access tokens into this notebook. Use only accounts and instructions supplied by your administrator. If the live demo could change data, use your team's designated test item and reset procedure.


## 10. Administrator setup checklist

This checklist explains what must exist without requiring the reader to perform API calls.

- Register the ProGear Sales Agent in Okta as an AI workload identity.
- Enable **direct User access** and assign the intended users or groups to the agent-bound sign-on application.
- Create the protected Inventory resource with separate `inventory:read` and `inventory:write` permissions.
- Configure the Inventory authorization policy so the sales group receives read access and the warehouse group receives read and write access.
- Ensure the application requests only the permission needed for the employee's action.
- Test both an allowed and denied case before demonstrating the scenario.
- Keep technical denial details in administrator logs, while returning plain-language guidance to the employee.
- Review the System Log to confirm the decision is attributable to both the employee and the ProGear Sales Agent.


## 11. Troubleshooting in plain language

| What you see | What it usually means | What to check |
|---|---|---|
| Everyone is denied | The policy or application assignment may be incomplete | Confirm User access assignments and the policy's allowed groups |
| Everyone can update inventory | The write rule may be too broad | Confirm only the warehouse group can receive `inventory:write` |
| A read question is denied | The application may be requesting write unnecessarily | Confirm the request asks only for `inventory:read` |
| A user changed groups but behavior did not change | The current session may contain older identity information | Sign out fully, sign in again, and retry |
| A denied response exposes technical names | Internal diagnostics are leaking into the user message | Keep those details in logs and use the approved plain-language response |


## 12. Glossary

- **Authentication:** Proving who a person or system is.
- **Authorization:** Deciding what that identity is allowed to do.
- **AI workload identity:** A distinct identity representing the AI agent, separate from a human user.
- **Direct User access:** The user signs in directly through the application bound to the registered AI agent.
- **Scope or permission:** A narrowly defined action such as reading or writing inventory.
- **ID-JAG:** A short-lived signed proof that binds the employee and agent identities for delegated access.
- **Least privilege:** Granting only the access necessary for a person's job and current request.

## Final takeaway

There is one ProGear Sales Agent. Sarah and Mike receive different outcomes because Okta evaluates **who is signed in**, **which agent is acting**, and **which permission the action requires**. A denied write is a successful security decision, and the employee receives a clear next step instead of internal technical details.
